# DeepSeek audio concept generation

This notebook walks through the three candidate sources used to form $\mathcal C_0 = \mathcal C_{LF} \cup \mathcal C_{broad} \cup \mathcal C_{contrast}$. DeepSeek only proposes candidates; the existing CLAP/LF-CBM pipeline determines which concepts are grounded and projectable.

Generation is checkpointed and resumable. API-spending cells are disabled by default.

## 1. Setup

Run from the repository root. Install the project requirements, then put 'DEEPSEEK_API_KEY=...' and optionally 'DEEPSEEK_MODEL=...' in a local '.env' file. Never commit that file.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

from dotenv import load_dotenv

import data_utils
from concept_generation_deepseek import (
    get_broad_prompt,
    get_contrastive_prompt,
    get_grouping_prompt,
    get_prompt,
)

load_dotenv()
print('DeepSeek key configured:', bool(os.getenv('DEEPSEEK_API_KEY')))

In [ ]:
dataset = 'urbansound8k'  # esc50 | urbansound8k | cremad
mode = 'all'             # lf | broad | contrastive | all
model = os.getenv('DEEPSEEK_MODEL', 'deepseek-v4-flash')
num_trials = 2
temperature = 0.4
output_dir = Path('data/concept_sets')

classes = data_utils.get_dataset_classes(dataset)
len(classes), classes, model

## 2. Inspect prompts before calling the API

The LF prompt is class-conditioned. The broad prompt never receives class names. Contrastive generation first discovers overlapping acoustic confusion groups, then asks for general audible dimensions jointly for each group.

In [ ]:
print('--- LF example ---')
print(get_prompt(dataset, 'important', classes[0]))
print('\n--- Broad vocabulary (excerpt) ---')
print(get_broad_prompt(num_concepts=80)[:1200])
print('\n--- Confusion-group discovery ---')
print(get_grouping_prompt(dataset, classes)[:1600])

In [ ]:
example_group = (
    ['drilling', 'jackhammer', 'engine_idling']
    if dataset == 'urbansound8k'
    else classes[:3]
)
print(get_contrastive_prompt(dataset, example_group, num_concepts=8))

## 3. Generate resumable candidate sets

Set 'RUN_GENERATION = True' when the prompts and settings look right. '--stage generate' calls DeepSeek but deliberately does not load CLAP. The shared broad checkpoint is generated once and reused across datasets. Omit '--restart' to resume existing checkpoints.

In [ ]:
RUN_GENERATION = False  # change to True to spend DeepSeek API credit

command = [
    sys.executable,
    '-m',
    'scripts.concepts.generate_deepseek_concept_sets',
    '--dataset', dataset,
    '--mode', mode,
    '--stage', 'generate',
    '--model', model,
    '--num-trials', str(num_trials),
    '--temperature', str(temperature),
]
print(' '.join(command))
if RUN_GENERATION:
    assert os.getenv('DEEPSEEK_API_KEY'), 'Set DEEPSEEK_API_KEY in .env first'
    subprocess.run(command, check=True)

The source files are written beneath 'data/concept_sets/<dataset>/':

- 'concepts_lf.txt'
- 'concepts_broad.txt'
- 'concepts_contrastive.txt'
- 'concepts_all.txt'
- 'concepts_metadata.json'
- 'contrastive_groups.json'

The text files are one concept per line and can be consumed by existing LF-CBM code.

In [ ]:
dataset_dir = output_dir / dataset
if dataset_dir.exists():
    for path in sorted(dataset_dir.glob('concepts_*.txt')):
        concepts = path.read_text(encoding='utf-8').splitlines()
        print(f'{path.name}: {len(concepts)} concepts')
        print('  ', concepts[:8])
else:
    print('No generated output yet:', dataset_dir)

In [ ]:
metadata_path = dataset_dir / 'concepts_metadata.json'
if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    print('Metadata records:', len(metadata))
    display(metadata[:8])

groups_path = dataset_dir / 'contrastive_groups.json'
if groups_path.exists():
    groups = json.loads(groups_path.read_text(encoding='utf-8'))['groups']
    print('Confusion groups:', len(groups))
    display(groups)

## 4. Reuse the existing LF-CBM text filters

Filtering is a separate stage. It removes overly long candidates, class-name-like candidates, and semantic duplicates using the repository's existing CLAP text space. Audio-activation grounding remains in CBM training, and projectability filtering remains after CBL learning.

In [ ]:
RUN_FILTERING = False  # can load/download the configured CLAP model

filter_command = [
    sys.executable,
    '-m',
    'scripts.concepts.generate_deepseek_concept_sets',
    '--dataset', dataset,
    '--mode', mode,
    '--stage', 'process',
    '--device', 'cuda',
]
print(' '.join(filter_command))
if RUN_FILTERING:
    subprocess.run(filter_command, check=True)

## 5. Ablations

| Variant | CLI mode | LF | Broad | Contrastive |
|---|---|---:|---:|---:|
| Audio LF-CBM | 'lf' | yes | no | no |
| + Broad | generate 'all', evaluate LF + broad metadata sources | yes | yes | no |
| + Contrastive | generate 'all', evaluate LF + contrastive metadata sources | yes | no | yes |
| Full | 'all' | yes | yes | yes |

The separate source text files make the two mixed ablations explicit: concatenate the desired source files with case-insensitive de-duplication, or select those sources from 'concepts_metadata.json'.

### Legacy baseline

For exact compatibility with the earlier script, omit '--mode' (or use '--mode legacy'). This retains the old '--datasets', LF prompt checkpoints, and '<dataset>_filtered_deepseek.txt' output path.